In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
# import geopandas as gpd
# from shapely.geometry import Point
from lisfloodutilities.cutmaps.cutlib import mask_from_ldd
from lisfloodutilities.nc2pcr import convert
from lisfloodutilities.readers import PCRasterMap

from tqdm import tqdm
import os
import shutil
import tempfile
import logging


In [21]:
# LDD direction convention (PCRaster):
# 7 8 9
# 4 5 6
# 1 2 3
# For each direction, what is the (row_offset, col_offset) of the downstream neighbour?
# e.g. direction 1 means flow goes to bottom-left → downstream is at (+1, -1)

# Inverse: for each neighbour position, which LDD value points TO the current cell?
# neighbour at (-1, -1) has direction 3 if it flows to us
# neighbour at (-1,  0) has direction 2
# neighbour at (-1, +1) has direction 1
# neighbour at ( 0, -1) has direction 6
# neighbour at ( 0, +1) has direction 4
# neighbour at (+1, -1) has direction 9
# neighbour at (+1,  0) has direction 8
# neighbour at (+1, +1) has direction 7

# (row_offset, col_offset) -> LDD value that points toward current cell
UPSTREAM_DIRS = {
    (-1, -1): 3,
    (-1,  0): 2,
    (-1, +1): 1,
    ( 0, -1): 6,
    ( 0, +1): 4,
    ( 1, -1): 9,
    ( 1,  0): 8,
    ( 1, +1): 7,
}

def cutmaps_own(ldd_array, lon_array, lat_array, station_lon, station_lat):
    """
    Trace upstream from station pixel through LDD to get catchment mask.
    Returns a 2D boolean array (same shape as ldd_array).
    """
    # find nearest pixel to station coordinates
    col = np.argmin(np.abs(lon_array - station_lon))
    row = np.argmin(np.abs(lat_array - station_lat))
    # print(f"  Station pixel: row={row}, col={col}, lon={lon_array[col]:.3f}, lat={lat_array[row]:.3f}")

    nrows, ncols = ldd_array.shape
    mask = np.zeros((nrows, ncols), dtype=bool)

    # BFS upstream flood-fill
    queue = [(row, col)]
    mask[row, col] = True

    while queue:
        r, c = queue.pop()
        for (dr, dc), upstream_val in UPSTREAM_DIRS.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < nrows and 0 <= nc < ncols:
                if not mask[nr, nc] and ldd_array[nr, nc] == upstream_val:
                    mask[nr, nc] = True
                    queue.append((nr, nc))

    return mask

# =============================================================================
# SEASONALITY HELPER
# =============================================================================

def seasonality_index(monthly_means: np.ndarray) -> float:
    """
    Walsh & Lawler (1981) seasonality index.
    monthly_means: array of 12 values (climatological monthly means)
    SI = (1/R) * sum(|xi - R/12|)  where R = annual sum
    Returns 0 (perfectly uniform) to ~1.83 (all rain in one month).
    """
    monthly_means = np.array(monthly_means, dtype=float)
    R = monthly_means.sum()
    if R == 0:
        return np.nan
    return float(np.sum(np.abs(monthly_means - R / 12)) / R)


def get_area_cut(area_array, area_lon, area_lat, bbox_lon, bbox_lat):
    ac_min = np.argmin(np.abs(area_lon - bbox_lon[0]))
    ac_max = np.argmin(np.abs(area_lon - bbox_lon[1]))
    ar_min = np.argmin(np.abs(area_lat - bbox_lat[0]))
    ar_max = np.argmin(np.abs(area_lat - bbox_lat[1]))
    return area_array[ar_min:ar_max+1, ac_min:ac_max+1]

In [16]:
# BASE_FILE = Path.cwd () / "GloFASv5_stations_metadata_calfunction_KGE_JSD_20March2026_final.csv"
BASE_FILE = Path.cwd () / "GloFASv5_catchments_comprehensive.csv"
# Folder with climate files
DIR_CLIM = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/long_term_runs/00_Climate/")
# Folder with static attribute files
DIR_STATIC = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/static_maps/GloFASv5_staticmaps_consolidated_March2026/GloFASv5_static_maps_reanalysis/")

# Base File
glofas5_base_info = pd.read_csv(BASE_FILE)


In [4]:
# station list
cal_stations = glofas5_base_info[["LISFLOOD_X","LISFLOOD_Y", "ID"]].values
with open(DIR_STATIC / "cal_stations.txt", "w") as f:
    for x, y, id_ in cal_stations:
        f.write(f"{x:.3f}  {y:.3f}\t{int(id_)}\n")

### Load ldd & uparea

In [ ]:
# Load LDD file
ldd_nc = DIR_STATIC / "ldd_repaired.nc"
ds_ldd = xr.open_dataset(ldd_nc)
print("\n=== ldd_repaired.nc ===")
print(ds_ldd)
print("\n=== Variables & dtypes ===")
for var in ds_ldd.data_vars:
    print(f"{var}: {ds_ldd[var].dtype}")
ds_ldd.close()

# Load upArea file
uparea_nc = DIR_STATIC / "upArea_repaired_correctedmetadata_3000.nc"
ds_uparea = xr.open_dataset(uparea_nc)
print("\n=== upArea_repaired_correctedmetadata_3000.nc ===")
print(ds_uparea)
print("\n=== Variables & dtypes ===")
for var in ds_uparea.data_vars:
    print(f"{var}: {ds_uparea[var].dtype}")
ds_uparea.close()


=== ldd_repaired.nc ===
<xarray.Dataset>
Dimensions:  (lon: 7200, lat: 3000)
Coordinates:
  * lon      (lon) float64 -180.0 -179.9 -179.9 -179.8 ... 179.9 179.9 180.0
  * lat      (lat) float64 89.97 89.92 89.88 89.82 ... -59.88 -59.92 -59.97
Data variables:
    crs      |S1 ...
    Band1    (lat, lon) float32 ...
Attributes:
    CDI:                        Climate Data Interface version 1.9.10 (https:...
    Conventions:                CF-1.5
    GDAL_AREA_OR_POINT:         Area
    GDAL_PCRASTER_VALUESCALE:   VS_LDD
    GDAL:                       GDAL 3.2.1, released 2020/12/29
    CDO:                        Climate Data Operators version 1.9.10 (https:...
    history:                    Fri Dec 10 11:49:28 2021: ncks -A -v lat,lon ...
    history_of_appended_files:  Fri Dec 10 11:49:28 2021: Appended file ldd_O...
    NCO:                        netCDF Operators version 4.9.7 (Homepage = ht...

=== Variables & dtypes ===
crs: |S1
Band1: float32

=== upArea_repaired_correctedmetad

### pythonic catchment masking

In [9]:
# ── load LDD once ─────────────────────────────────────────────────────────────
ds_ldd = xr.open_dataset(DIR_STATIC / "ldd_repaired.nc")
ldd_array = ds_ldd['Band1'].values
lon_array = ds_ldd.lon.values
lat_array = ds_ldd.lat.values

# ── test on first station ─────────────────────────────────────────────────────
x, y, station_id = cal_stations[0]
station_id = int(station_id)
print(f"Testing station {station_id} at ({x}, {y})")

import time
t0 = time.time()
catchment_mask = cutmaps_own(ldd_array, lon_array, lat_array, x, y)
print(f"Done in {time.time()-t0:.2f}s")
print(f"Catchment size: {catchment_mask.sum()} pixels")

Testing station 4 at (11.875, 52.025)
Done in 0.05s
Catchment size: 4747 pixels


In [ ]:
plt.pcolormesh(catchment_mask)

### loop over all catchments

In [ ]:
"""
catchment_meteo_attributes.py
------------------------------
Extracts catchment-average climate statistics for all GloFAS calibration
stations from yearly global NetCDF files (tp, eT0, ta).

Strategy:
  1. Compute BFS catchment masks for all stations once (on LDD)
  2. Loop over years x variables — load one file at a time
  3. For each station: slice bbox, apply mask, compute stats
  4. Aggregate across years -> long-term means, interannual std, seasonality, aridity index
"""

import time
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from tqdm import tqdm

# =============================================================================
# CONFIG
# =============================================================================

STATIC_DIR  = DIR_STATIC
CLIMATE_DIR = DIR_CLIM  # root dir where glofas_eT0_1999.nc etc. live

LDD_NC     = STATIC_DIR / "ldd_repaired.nc"
PIXAREA_NC = STATIC_DIR / "pixarea_Global_03min.nc"

CLIM_VARS  = ["tp", "eT0", "ta"]
YEARS      = range(1991, 2021)  # adjust to your available years

def get_clim_path(var: str, year: int) -> Path:
    return CLIMATE_DIR / f"glofas_{var}_{year}.nc"


# =============================================================================
# STEP 0 — load LDD + pixel area, compute all BFS masks
# =============================================================================

print("Loading LDD...")
ds_ldd    = xr.open_dataset(LDD_NC)
ldd_array = np.squeeze(ds_ldd['Band1'].values)
lon_array = ds_ldd.lon.values
lat_array = ds_ldd.lat.values
print(f"LDD shape: {ldd_array.shape}")

print("Loading pixel area...")
ds_area    = xr.open_dataset(PIXAREA_NC)
area_array = np.squeeze(ds_area[list(ds_area.data_vars)[-1]].values)
area_lon   = ds_area.lon.values
area_lat   = ds_area.lat.values

print("Computing BFS catchment masks for all stations...")
cal_stations = glofas5_base_info[["LISFLOOD_X", "LISFLOOD_Y", "ID"]].values

station_masks = {}  # station_id -> (mask_cut, r_min, r_max, c_min, c_max) or None
t0 = time.time()
for x, y, station_id in tqdm(cal_stations, desc="BFS masks"):
    station_id = int(station_id)
    mask = cutmaps_own(ldd_array, lon_array, lat_array, x, y)
    rows_idx, cols_idx = np.where(mask)
    if len(rows_idx) == 0:
        station_masks[station_id] = None
        continue
    r_min, r_max = rows_idx.min(), rows_idx.max()
    c_min, c_max = cols_idx.min(), cols_idx.max()
    mask_cut = mask[r_min:r_max+1, c_min:c_max+1]
    station_masks[station_id] = (mask_cut, r_min, r_max, c_min, c_max)

print(f"All masks done in {time.time()-t0:.1f}s")


# =============================================================================
# STEP 1 — loop variables x years, accumulate per station
# =============================================================================

# accumulator: results_accum[station_id][var] holds annual values + monthly climatology
results_accum = {
    int(sid): {
        var: {
            "annual":         [],            # annual sum (tp/eT0) or mean (ta)
            "monthly_sums":   np.zeros(12),  # accumulated monthly values across years
            "monthly_counts": np.zeros(12),  # number of years contributing per month
        }
        for var in CLIM_VARS
    }
    for _, _, sid in cal_stations
}

for var in CLIM_VARS:
    for year in tqdm(YEARS, desc=f"{var}"):
        fpath = get_clim_path(var, year)
        if not fpath.exists():
            print(f"  WARNING: missing {fpath.name}, skipping")
            continue

        # lazy open — nothing loaded into memory yet
        ds = xr.open_dataset(fpath)
        da = ds[var]  # (time, lat, lon)
        clim_lon = da.lon.values
        clim_lat = da.lat.values
        months   = da.time.dt.month.values  # (n_timesteps,) ints 1-12

        for x, y, station_id in cal_stations:
            station_id = int(station_id)
            entry = station_masks[station_id]
            if entry is None:
                continue

            mask_cut, r_min, r_max, c_min, c_max = entry

            bbox_lon = lon_array[[c_min, c_max]]
            bbox_lat = lat_array[[r_min, r_max]]

            # map bbox corners to climate grid indices
            ci_min = np.argmin(np.abs(clim_lon - bbox_lon[0]))
            ci_max = np.argmin(np.abs(clim_lon - bbox_lon[1]))
            ri_min = np.argmin(np.abs(clim_lat - bbox_lat[0]))
            ri_max = np.argmin(np.abs(clim_lat - bbox_lat[1]))

            # load only the bbox slice — this is the memory-safe key step
            da_cut = da.isel(
                lat=slice(ri_min, ri_max+1),
                lon=slice(ci_min, ci_max+1)
            ).values.astype(float)  # (time, H, W)

            if da_cut.shape[1:] != mask_cut.shape:
                continue

            # spatial aggregation -> catchment timeseries (n_timesteps,)
            pixels = da_cut[:, mask_cut]  # (time, n_pixels)

            area_cut = get_area_cut(area_array, area_lon, area_lat, bbox_lon, bbox_lat)
            if area_cut.shape == mask_cut.shape:
                w     = area_cut[mask_cut].astype(float)
                w     = np.where(np.isfinite(w), w, 0.0)
                w_sum = w.sum()
                ts    = np.sum(pixels * w[np.newaxis, :], axis=1) / w_sum
            else:
                ts = np.nanmean(pixels, axis=1)

            # --- annual value ---
            if var == "ta":
                annual_val = float(np.nanmean(ts))
            else:  # tp, eT0 -> annual sum
                annual_val = float(np.nansum(ts))
            results_accum[station_id][var]["annual"].append(annual_val)

            # --- monthly climatology ---
            for m in range(1, 13):
                idx = months == m
                if idx.sum() == 0:
                    continue
                if var == "ta":
                    monthly_val = float(np.nanmean(ts[idx]))
                else:
                    monthly_val = float(np.nansum(ts[idx]))
                results_accum[station_id][var]["monthly_sums"][m-1]   += monthly_val
                results_accum[station_id][var]["monthly_counts"][m-1] += 1

        ds.close()


# =============================================================================
# STEP 2 — aggregate years -> one row per station
# =============================================================================

results = []
for _, _, station_id in cal_stations:
    station_id = int(station_id)
    row = {"ID": station_id}

    tp_annual_mean  = np.nan
    et0_annual_mean = np.nan

    for var in CLIM_VARS:
        acc    = results_accum[station_id][var]
        annual = [v for v in acc["annual"] if np.isfinite(v)]

        # climatological monthly means
        monthly_clim = np.where(
            acc["monthly_counts"] > 0,
            acc["monthly_sums"] / acc["monthly_counts"],
            np.nan
        )  # shape (12,)

        if var == "ta":
            row["ta_mean"]         = float(np.mean(annual))        if annual else np.nan
            row["ta_std_interann"] = float(np.std(annual, ddof=1)) if len(annual) > 1 else np.nan
            row["ta_seasonality"]  = seasonality_index(monthly_clim)

        else:  # tp, eT0
            row[f"{var}_mean_annual"]  = float(np.mean(annual))        if annual else np.nan
            row[f"{var}_std_interann"] = float(np.std(annual, ddof=1)) if len(annual) > 1 else np.nan
            row[f"{var}_seasonality"]  = seasonality_index(monthly_clim)

            if var == "tp":
                tp_annual_mean  = row["tp_mean_annual"]
            if var == "eT0":
                et0_annual_mean = row["eT0_mean_annual"]

    # aridity index PET/P — >1 arid, <1 humid
    if np.isfinite(tp_annual_mean) and tp_annual_mean > 0:
        row["aridity_index"] = float(et0_annual_mean / tp_annual_mean)
    else:
        row["aridity_index"] = np.nan

    results.append(row)


# =============================================================================
# STEP 3 — merge into glofas5_base_info and save
# =============================================================================

df_clim = pd.DataFrame(results)
df_clim["ID"] = df_clim["ID"].astype(int)
glofas5_base_info["ID"] = glofas5_base_info["ID"].astype(int)

glofas5_enriched_clim = glofas5_base_info.merge(df_clim, on="ID", how="left")

print(f"Final table shape: {glofas5_enriched_clim.shape}")
print(glofas5_enriched_clim.head())

# glofas5_enriched_clim.to_csv("glofas5_catchment_meteo_attributes.csv", index=False)
# glofas5_enriched_clim.to_parquet("glofas5_catchment_meteo_attributes.parquet", index=False)
print("\nDone.")

Loading LDD...
LDD shape: (3000, 7200)
Loading pixel area...
Computing BFS catchment masks for all stations...


BFS masks:  80%|███████▉  | 4279/5379 [05:39<00:56, 19.63it/s]

In [ ]:
# save the results
glofas5_enriched_clim.to_csv("GloFASv5_Stations_Overview_260223/GloFASv5_catchments_clim_comprehensive.csv", index=False)


## OLD:
### compare cutmaps_own with lisflood utilities cutmaps

In [ ]:
# Test station
x, y, station_id = cal_stations[100]
station_id = int(station_id)
print(f"Testing station {station_id} at ({x}, {y})")

# ── Mask 1: our BFS approach ──────────────────────────────────────────────────
t0 = time.time()
mask_bfs = cutmaps_own(ldd_array, lon_array, lat_array, x, y)
print(f"BFS done in {time.time()-t0:.2f}s, pixels: {mask_bfs.sum()}")

# ── Mask 2: cutmaps/mask_from_ldd ────────────────────────────────────────────
import tempfile, shutil
from lisfloodutilities.cutmaps.cutlib import mask_from_ldd

t0 = time.time()
with tempfile.TemporaryDirectory() as stmp:
    station_txt = Path(stmp) / "station.txt"
    np.savetxt(station_txt, [[x, y, station_id]], fmt="%.3f  %.3f\t%d")
    mask, outlets_nc, maskmap_nc = mask_from_ldd(str(LDD_MAP), str(station_txt))
    ds_cutmaps = xr.open_dataset(maskmap_nc)
    # keep in memory before temp dir closes
    mask_cutmaps_data = ds_cutmaps['area'].values  # 1 inside, 0 outside
    mask_cutmaps_lon  = ds_cutmaps.x.values
    mask_cutmaps_lat  = ds_cutmaps.y.values
print(f"cutmaps done in {time.time()-t0:.2f}s, pixels: {mask_cutmaps_data.sum()}")

# ── compare pixel counts ──────────────────────────────────────────────────────
# get bounding box from cutmaps result and extract same region from BFS mask
col_min = np.argmin(np.abs(lon_array - mask_cutmaps_lon.min()))
col_max = np.argmin(np.abs(lon_array - mask_cutmaps_lon.max()))
row_min = np.argmin(np.abs(lat_array - mask_cutmaps_lat.max()))  # lat is descending
row_max = np.argmin(np.abs(lat_array - mask_cutmaps_lat.min()))

mask_bfs_cut = mask_bfs[row_min:row_max+1, col_min:col_max+1].astype(np.int8)
mask_cutmaps_binary = (mask_cutmaps_data == 1).astype(np.int8)  # NaN → 0, 1 → 1

print(f"Pixel count — BFS: {mask_bfs.sum()}, cutmaps: {mask_cutmaps_binary.sum()}")
print(f"BFS in bbox: {mask_bfs_cut.sum()}, cutmaps in bbox: {mask_cutmaps_binary.sum()}")
print(f"Identical: {np.array_equal(mask_bfs_cut, mask_cutmaps_binary)}")
print(f"Overlap (IoU): {np.logical_and(mask_bfs_cut, mask_cutmaps_binary).sum() / np.logical_or(mask_bfs_cut, mask_cutmaps_binary).sum():.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(mask_bfs_cut, origin='upper', cmap='Blues')
axes[0].set_title(f'BFS ({mask_bfs_cut.sum()} px)')
axes[1].imshow(mask_cutmaps_binary, origin='upper', cmap='Blues')
axes[1].set_title(f'cutmaps ({mask_cutmaps_binary.sum()} px)')
diff = mask_bfs_cut.astype(int) - mask_cutmaps_binary.astype(int)
axes[2].imshow(diff, origin='upper', cmap='RdBu', vmin=-1, vmax=1)
axes[2].set_title('Difference (red=BFS only, blue=cutmaps only)')
plt.tight_layout()

### get catchment stats via cutmaps & catchstats

In [ ]:
# test = ds_uparea.sel(lat=glofas5_base_info.LISFLOOD_Y.loc[1], lon=glofas5_base_info.LISFLOOD_X.loc[1])
stations = DIR_STATIC / "stations.txt"
ldd_nc = DIR_STATIC / "ldd_repaired.nc"
ldd_pcr = DIR_STATIC / "ldd_repaired.map"
# clonemap = DIR_STATIC / "ldd_repaired.map'
# ldd_pcr = convert(ldd_nc, clonemap, 'tests/data/cutmaps/ldd_eu_test.map', is_ldd=True)[0]

# mask, outlets_nc, maskmap_nc = mask_from_ldd(ldd_pcr, stations)


In [ ]:
print("mask lon:", ds_mask.lon.values[:3])
print("elv  lon:", xr.open_dataset(get_attr_path("elv")).lon.values[:3])

In [ ]:
import tempfile
import shutil
import numpy as np
import xarray as xr
from pathlib import Path
from lisfloodutilities.cutmaps.cutlib import mask_from_ldd
from lisfloodutilities.catchstats import catchment_statistics

STATIC_DIR = Path(
    "/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5"
    "/static_maps/GloFASv5_staticmaps_consolidated_March2026"
    "/GloFASv5_static_maps_reanalysis"
)
LDD_MAP    = STATIC_DIR / "ldd_repaired.map"
PIXAREA_NC = STATIC_DIR / "pixarea_Global_03min.nc"

def get_attr_path(attr: str) -> Path:
    if attr in ("laii", "laif"):
        return STATIC_DIR / f"{attr}.nc"
    return STATIC_DIR / f"{attr}_Global_03min.nc"

# ── pick one station ──────────────────────────────────────────────────────────
x, y, station_id = cal_stations[0]
station_id = int(station_id)
print(f"Testing station {station_id} at ({x}, {y})")

# ── STEP 1: generate mask ─────────────────────────────────────────────────────
with tempfile.TemporaryDirectory() as stmp:
    station_txt = Path(stmp) / "station.txt"
    np.savetxt(station_txt, [[x, y, station_id]], fmt="%.3f  %.3f\t%d")
    mask, outlets_nc, maskmap_nc = mask_from_ldd(str(LDD_MAP), str(station_txt))
    print(f"maskmap_nc: {maskmap_nc}")
    dest = Path(f"./test_mask_{station_id}.nc")
    shutil.copy(maskmap_nc, dest)

# ── STEP 2: load mask ─────────────────────────────────────────────────────────
ds_mask = xr.open_dataset(dest)
ds_mask = ds_mask.rename({'x': 'lon', 'y': 'lat'})
print(ds_mask)
var = list(ds_mask.data_vars)[0]
masks = {station_id: ds_mask[var]}

# ── STEP 3 + 4: catchstats per attribute ─────────────────────────────────────
TEST_ATTRS = ["elv", "gradient"]

pixarea_ds = xr.open_dataset(PIXAREA_NC)
pixarea_da = pixarea_ds[list(pixarea_ds.data_vars)[0]]

stats_list = []
for attr in TEST_ATTRS:
    path = get_attr_path(attr)
    ds = xr.open_dataset(path)
    var = list(ds.data_vars)[0]
    ds = ds.rename({var: attr})
    print(f"  Running catchstats for {attr} ← {path.name}")

    ds_stats = catchment_statistics(
        ds,
        masks,
        statistic=["mean", "median"],
        weight=pixarea_da,
        output=None,
    )
    stats_list.append(ds_stats)

# ── STEP 5: combine + check ───────────────────────────────────────────────────
ds_all = xr.merge(stats_list, compat='override')
print(ds_all)

df_stats = ds_all.to_dataframe().reset_index()
print(df_stats)

In [ ]:
"""
catchment_attributes.py
-----------------------
Extracts catchment-average attributes for all GloFAS calibration stations.

Step 0: write cal_stations.txt from glofas5_base_info
Step 1: mask_from_ldd loop → one mask .nc per station saved to MASKS_DIR/{ID}.nc
Step 2: catchstats → mean + median of all attributes over each catchment mask
Step 3: merge results into glofas5_base_info and save

Assumes glofas5_base_info is already loaded (e.g. earlier in your notebook).
"""

import shutil
import tempfile
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from tqdm import tqdm

from lisfloodutilities.cutmaps.cutlib import mask_from_ldd
from lisfloodutilities.catchstats import catchment_statistics

# =============================================================================
# CONFIG
# =============================================================================

STATIC_DIR = Path(
    "/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5"
    "/static_maps/GloFASv5_staticmaps_consolidated_March2026"
    "/GloFASv5_static_maps_reanalysis"
)

LDD_MAP    = STATIC_DIR / "ldd_repaired.map"     # already in PCRaster format — no conversion needed
PIXAREA_NC = STATIC_DIR / "pixarea_Global_03min.nc"

MASKS_DIR    = Path("./masks")
STATIONS_TXT = Path("./cal_stations.txt")

ATTRIBUTES = [
    "elv",
    "laii",                        # → laii.nc (no suffix)
    "laif",                        # → laif.nc (no suffix)
    "gradient",
    "lusemask",
    "outlets_calibratedstations",
    "soildepth1_f",
    "soildepth1_o",
    "fracforest",
    "fracirrigated",
    "fracother",
    "ksat1_f",
    "ksat1_o",
]

def get_attr_path(attr: str) -> Path:
    if attr in ("laii", "laif"):
        return STATIC_DIR / f"{attr}.nc"
    return STATIC_DIR / f"{attr}_Global_03min.nc"

# =============================================================================
# STEP 1 — mask_from_ldd: generate one mask .nc per station
# =============================================================================

MASKS_DIR.mkdir(parents=True, exist_ok=True)

failed = []
for x, y, station_id in tqdm(cal_stations, desc="Generating masks"):
    station_id = int(station_id)
    mask_out = MASKS_DIR / f"{station_id}.nc"

    if mask_out.exists():
        continue  # safe to rerun — skips already completed stations

    with tempfile.TemporaryDirectory() as stmp:
        station_txt = Path(stmp) / "station.txt"
        np.savetxt(station_txt, [[x, y, station_id]], fmt="%.3f  %.3f\t%d")

        try:
            mask, outlets_nc, maskmap_nc = mask_from_ldd(str(LDD_MAP), str(station_txt))

            if maskmap_nc and Path(maskmap_nc).exists():
                shutil.copy(maskmap_nc, mask_out)
            else:
                print(f"  WARNING: no maskmap_nc for station {station_id}")
                failed.append(station_id)

        except Exception as e:
            print(f"  ERROR station {station_id}: {e}")
            failed.append(station_id)

mask_files = sorted(MASKS_DIR.glob("*.nc"))
print(f"\nMasks generated: {len(mask_files)} / {len(cal_stations)}")
if failed:
    print(f"Failed stations: {failed}")

# =============================================================================
# STEP 2 — catchstats: compute mean + median per catchment
# =============================================================================

# Load masks: Dict[int, xarray.DataArray]
masks = {}
for p in mask_files:
    ds = xr.open_dataset(p)
    var = list(ds.data_vars)[0]
    masks[int(p.stem)] = ds[var]
print(f"Loaded {len(masks)} masks")

# Load + merge all attribute files into one xarray.Dataset
attr_datasets = []
for attr in ATTRIBUTES:
    path = get_attr_path(attr)
    if not path.exists():
        print(f"  WARNING: not found, skipping → {path}")
        continue
    ds = xr.open_dataset(path)
    var = list(ds.data_vars)[0]
    attr_datasets.append(ds.rename({var: attr}))
    print(f"  Loaded {attr} ← {path.name}")

maps = xr.merge(attr_datasets)
print(f"\nMerged attribute dataset: {list(maps.data_vars)}")

# Pixel area for area-weighted stats (important for global 3-arcmin grid)
pixarea_ds = xr.open_dataset(PIXAREA_NC)
pixarea_da = pixarea_ds[list(pixarea_ds.data_vars)[0]]

# Run catchstats — fully in memory
print("\nRunning catchstats...")
ds_stats = catchment_statistics(
    maps,
    masks,
    statistic=["mean", "median"],
    weight=pixarea_da,
    output=None,
)
print("Done.")
print(ds_stats)

# =============================================================================
# STEP 3 — merge into glofas5_base_info and save
# =============================================================================

df_stats = ds_stats.to_dataframe().reset_index()
df_stats["ID"] = df_stats["ID"].astype(int)
glofas5_base_info["ID"] = glofas5_base_info["ID"].astype(int)

glofas5_enriched = glofas5_base_info.merge(df_stats, on="ID", how="left")

print(f"\nFinal table: {glofas5_enriched.shape}")
print(glofas5_enriched.head())

glofas5_enriched.to_csv("glofas5_catchment_attributes.csv", index=False)
glofas5_enriched.to_parquet("glofas5_catchment_attributes.parquet", index=False)
print("\nSaved → glofas5_catchment_attributes.csv / .parquet")

Generating masks:   0%|          | 0/5379 [00:00<?, ?it/s]col2map version: 4.4.0 (linux/x86_64)
nr. of records read: 1
nr. of records with mv value: 0
nr. of records with mv (x,y): 0
nr. of records outside map: 0
nr. of cells with mv: 21599999
nr. of cells with more than one record: 0
nr. of cells with majority conflict: 0
[14:21:09] - Defining WGS84 coordinates variables
[14:21:09] - Creating NETCDF3_CLASSIC main variable outlets: type int32 fill value 0 (<class 'int'>)
[14:21:10] - Adding /tmp/tmpwmwnmh_k/outlets.map - timestep None - hour 0.0
[14:21:10] - Writing /tmp/tmpwmwnmh_k/outlets.nc
col2map version: 4.4.0 (linux/x86_64)
nr. of records read: 1
nr. of records with mv value: 0
nr. of records with mv (x,y): 0
nr. of records outside map: 0
nr. of cells with mv: 21599999
nr. of cells with more than one record: 0
nr. of cells with majority conflict: 0
pcrcalc version: 4.4.0 (linux/x86_64)
pcrcalc version: 4.4.0 (linux/x86_64)
?:1:52:ERROR: /tmp/tmpwmwnmh_k/catchmask00079.map: File 

  ERROR station 79: [Errno 2] No such file or directory: '/tmp/tmpwmwnmh_k/catchmask00079.map'


nr. of cells with mv: 21599999
nr. of cells with more than one record: 0
nr. of cells with majority conflict: 0
[14:21:12] - Defining WGS84 coordinates variables
[14:21:12] - Creating NETCDF3_CLASSIC main variable outlets: type int32 fill value 0 (<class 'int'>)
[14:21:12] - Adding /tmp/tmpf5hb82lk/outlets.map - timestep None - hour 0.0
[14:21:13] - Writing /tmp/tmpf5hb82lk/outlets.nc
col2map version: 4.4.0 (linux/x86_64)
nr. of records read: 1
nr. of records with mv value: 0
nr. of records with mv (x,y): 0
nr. of records outside map: 0
nr. of cells with mv: 21599999
nr. of cells with more than one record: 0
nr. of cells with majority conflict: 0
pcrcalc version: 4.4.0 (linux/x86_64)
pcrcalc version: 4.4.0 (linux/x86_64)
pcrcalc version: 4.4.0 (linux/x86_64)
pcrcalc version: 4.4.0 (linux/x86_64)
pcrcalc version: 4.4.0 (linux/x86_64)
pcrcalc version: 4.4.0 (linux/x86_64)
resample version: 4.4.0 (linux/x86_64)
